# Data Drift Analysis

Compare the current production traffic of the credit-scoring API to the training reference set using [Evidently](https://docs.evidentlyai.com/). The notebook is reproducible: if a `DATABASE_URL` is configured it pulls real prediction logs from Supabase, otherwise it synthesizes a drifted batch from the reference data so the analysis can be demonstrated offline.

Outputs:
- `docs/drift_report.html` — full interactive Evidently report
- a narrative summary at the bottom of this notebook

In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.preprocessing import build_feature_dataframe  # noqa: E402
from monitoring.drift_detection import (  # noqa: E402
    generate_drift_report,
    load_reference_data,
    save_drift_report_html,
)

## 1. Reference data

Loaded from `monitoring/reference_data.csv` — a 1000-row sample of the training set used in Part 1, aligned with the 100 features the model expects.

In [ ]:
reference = load_reference_data()
print(f"reference shape: {reference.shape}")
reference.head(3)

## 2. Production data

If `DATABASE_URL` is set the notebook pulls the most recent prediction logs from Supabase and projects each API input into the same 100-feature space using `build_feature_dataframe`. Otherwise we synthesize a deliberately drifted batch from the reference: income shifted +20%, larger missingness in `EXT_SOURCE_2`, and a subsample of 500 rows. This matches the kind of distribution shift we would expect if a new originating channel started routing higher-income applicants to the model.

In [ ]:
def load_production_from_db(limit: int = 1000) -> pd.DataFrame | None:
    if not os.environ.get("DATABASE_URL"):
        return None
    from app.services.db_service import get_recent_predictions
    rows = get_recent_predictions(limit=limit)
    if not rows:
        return None
    frames = [build_feature_dataframe(dict(r["input_features"])) for r in rows]
    return pd.concat(frames, ignore_index=True)


def synthesize_drift(reference: pd.DataFrame, rng_seed: int = 7) -> pd.DataFrame:
    rng = np.random.default_rng(rng_seed)
    sample = reference.sample(n=500, random_state=rng_seed).copy()
    if "AMT_INCOME_TOTAL" in sample.columns:
        sample["AMT_INCOME_TOTAL"] = sample["AMT_INCOME_TOTAL"] * 1.20
    if "AMT_CREDIT" in sample.columns:
        sample["AMT_CREDIT"] = sample["AMT_CREDIT"] * 1.10
    if "DAYS_BIRTH" in sample.columns:
        sample["DAYS_BIRTH"] = sample["DAYS_BIRTH"] - 365 * 3
    if "EXT_SOURCE_2" in sample.columns:
        mask = rng.random(len(sample)) < 0.3
        sample.loc[mask, "EXT_SOURCE_2"] = np.nan
    return sample.reset_index(drop=True)


production = load_production_from_db()
if production is None:
    print("DATABASE_URL not set or no rows in DB. Using synthesized drift batch.")
    production = synthesize_drift(reference)
else:
    print(f"Loaded {len(production)} rows from Supabase.")

print(f"production shape: {production.shape}")
production.head(3)

## 3. Evidently drift report

`generate_drift_report` runs the `DataDriftPreset` over every column that has at least one non-null value in both frames. The HTML is saved to `docs/drift_report.html` so it can be linked from the README.

In [ ]:
snapshot = generate_drift_report(reference, production)
report_path = PROJECT_ROOT / "docs" / "drift_report.html"
report_path.parent.mkdir(parents=True, exist_ok=True)
save_drift_report_html(snapshot, str(report_path))
print(f"Drift report saved to {report_path}")

In [ ]:
html = snapshot.get_html_str(as_iframe=False)
display(HTML(html))

## 4. Findings and retraining policy

**What to look at in the Evidently report above:**
- The summary banner reports the share of drifted columns. Evidently uses Wasserstein distance for numerical features and Jensen–Shannon for categoricals (defaults), flagging a column when the test statistic crosses an automatically scaled threshold.
- Per-feature panels expose the reference vs. production distribution. Watch `AMT_INCOME_TOTAL`, `AMT_CREDIT`, `DAYS_BIRTH`, `EXT_SOURCE_2`/`EXT_SOURCE_3`, and the `PAYMENT_RATE` / `INCOME_CREDIT_PERC` engineered features — these are the highest-importance inputs to the LightGBM model, so drift there is most likely to degrade scoring quality.

**Retraining thresholds we propose:**
1. **More than 30% of monitored columns drifted** in any 7-day window → schedule a retraining sprint and refresh `monitoring/reference_data.csv` after the new model ships.
2. **Any single top-importance feature drifted** (the four `EXT_SOURCE_*` and `PAYMENT_RATE` columns) → page the data team within 48 h; investigate whether the upstream source changed before retraining.
3. **Approval rate moves by more than ±5 pp** versus the trailing 30-day baseline (visible on the Streamlit dashboard) → cross-check against drift and against the cost matrix used to set `OPTIMAL_THRESHOLD = 0.0874`.

**Operational notes:**
- The synthesized drift here is intentionally aggressive (+20% income, -3 yr birth, 30% NaN injected in `EXT_SOURCE_2`). Real production drift will usually be smaller and slower; the Streamlit dashboard surfaces the same report on real Supabase data when the API has accumulated enough traffic.
- Engineered features (`PAYMENT_RATE`, `INCOME_CREDIT_PERC`, `ANNUITY_INCOME_PERC`) inherit drift from their raw components, so they should not be treated as independent signals.
- 71 of the 100 features are aggregations from auxiliary tables (bureau, previous applications) that the public API does not collect. They sit as NaN in production rows and are imputed at inference; drift detection on those columns is uninformative and is automatically skipped by `generate_drift_report`.